# 02 · Chuẩn bị dữ liệu MBD-mini cho propensity
Notebook chứa đầy đủ mã xử lí và các bước thực thi riêng, dùng cùng quy tắc với package `propensity_data`. Không cần import package nội bộ để chạy notebook.

**Trạng thái bàn giao: chưa thực thi.** Mọi code cell có `execution_count=null`, không có output; chưa tạo feature, fit mô hình hoặc chạy kiểm thử trên máy hiện tại.

Đơn vị dữ liệu là **khách–tháng**. Nhãn `y_any` cho biết mua ít nhất một trong bốn sản phẩm có nhãn. Giữ khách thiếu nguồn; dòng mất cả bốn target không dùng để train/evaluate, vẫn giữ để chấm điểm.

Luồng: cấu hình → quy tắc nhãn → chuẩn hóa từng nguồn → feature trước cutoff → kiểm kê → bàn giao cho mô hình.


## 1. Môi trường trên máy thực thi
Dùng Python >=3.11. Tại thư mục gốc repo, cài một lần trong terminal:
```powershell
.\.venv\Scripts\python.exe -m pip install -e ".[prepare,notebook]"
```
Chọn đúng kernel Python của môi trường đã cài. Notebook **không tự cài dependency**. Chạy cell theo thứ tự; output phải là thư mục mới. Nếu một bước lỗi, manifest chuyển `failed`; sửa nguyên nhân, đổi output và Restart Kernel trước khi chạy lại.

Dữ liệu thật nằm trong `data/MBD-mini/`, gồm `client_split`, `targets` và `detail/{trx,geo,dialog}`. Notebook xem mẫu `01_data_preview.ipynb` dùng sample riêng; sample này không dùng để đánh giá mô hình.


In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager
import hashlib
import json

import duckdb
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
from IPython.display import display

if sys.version_info < (3, 11):
    raise RuntimeError("Cần Python >=3.11.")

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục notebooks/ hoặc trong repo.")
print(PROJECT_ROOT)


## 2. Cấu hình và danh sách biến
Cell dưới định nghĩa cấu hình và kiểm tra đường dẫn/cửa sổ/tháng. Đường dẫn trong JSON tính từ file JSON. Mặc định dùng DuckDB 6 GB và hai luồng; giới hạn engine không bảo đảm tổng RSS tiến trình. Embedding tắt mặc định.

`mon` thường là ngày cuối tháng; cutoff là đầu **cùng tháng** lúc 00:00 UTC. Không dịch thêm một tháng hoặc ba giờ.


In [ ]:
"""Validated configuration; all relative paths are relative to the JSON file."""
from dataclasses import asdict, dataclass
from datetime import date
import json
from pathlib import Path
import re


def month_start(value: str) -> date:
    if not isinstance(value, str) or not re.fullmatch(r"\d{4}-\d{2}", value):
        raise ValueError("Months must use YYYY-MM")
    return date.fromisoformat(value + "-01")


@dataclass(frozen=True)
class PreparationConfig:
    data_root: Path
    output_dir: Path
    first_month: str = "2022-02"
    last_month: str = "2023-01"
    windows_days: tuple[int, ...] = (30, 84)
    memory_limit: str = "6GB"
    threads: int = 2
    label_delay_days: int = 0
    calibration_month: str = "2022-12"
    test_month: str = "2023-01"
    validation_months: tuple[str, ...] = ("2022-09", "2022-10", "2022-11")
    include_dialog_embeddings: bool = False
    embedding_size: int = 768
    embedding_window_days: int = 84

    def __post_init__(self):
        for key in ("data_root", "output_dir"):
            object.__setattr__(self, key, Path(getattr(self, key)).resolve())
        for key in ("windows_days", "validation_months"):
            object.__setattr__(self, key, tuple(getattr(self, key)))
        if (self.output_dir == self.data_root or self.data_root in self.output_dir.parents
                or self.output_dir in self.data_root.parents):
            raise ValueError("Output and raw-data trees must not overlap")
        for key in ("threads", "embedding_size", "embedding_window_days"):
            value = getattr(self, key)
            if type(value) is not int or value < 1:
                raise ValueError(f"{key} must be a positive integer")
        if type(self.label_delay_days) is not int or self.label_delay_days < 0:
            raise ValueError("label_delay_days must be a nonnegative integer")
        if type(self.include_dialog_embeddings) is not bool:
            raise ValueError("include_dialog_embeddings must be boolean")
        if not self.windows_days or any(type(w) is not int or w < 1 for w in self.windows_days):
            raise ValueError("windows_days must contain positive integers")
        if len(set(self.windows_days)) != len(self.windows_days):
            raise ValueError("Duplicate windows")
        if self.include_dialog_embeddings and self.embedding_window_days not in self.windows_days:
            raise ValueError("embedding_window_days must occur in windows_days")
        if not isinstance(self.memory_limit, str) or not re.fullmatch(r"[1-9]\d*(MB|GB)", self.memory_limit):
            raise ValueError("memory_limit must be a positive integer followed by MB or GB")
        months = [self.first_month, self.last_month, self.calibration_month, self.test_month,
                  *self.validation_months]
        for value in months:
            month_start(value)
        if not self.first_month <= self.calibration_month < self.test_month <= self.last_month:
            raise ValueError("Require first <= calibration < test <= last")
        if (not self.validation_months or tuple(sorted(set(self.validation_months))) != self.validation_months
                or not all(self.first_month < m < self.calibration_month for m in self.validation_months)):
            raise ValueError("Validation months must be ordered, unique, and before calibration")

    def to_dict(self):
        result = asdict(self)
        for key in ("data_root", "output_dir"):
            result[key] = str(result[key])
        return result


def load_config(path: str | Path) -> PreparationConfig:
    path = Path(path).resolve()
    options = json.loads(path.read_text(encoding="utf-8-sig"))
    for key in ("data_root", "output_dir"):
        options[key] = (path.parent / options[key]).resolve()
    return PreparationConfig(**options)


In [ ]:
"""Source layout and anonymized MBD-mini columns."""
SOURCES = {"trx": "detail/trx", "geo": "detail/geo", "dialog": "detail/dialog"}
LABELS = tuple(f"target_{i}" for i in range(1, 5))
CATEGORIES = ("event_type", "event_subtype", "src_type11", "src_type12", "dst_type11",
              "dst_type12", "src_type21", "src_type22", "src_type31", "src_type32")


def feature_schema(columns: list[str]) -> dict:
    """Explicit whitelist; never expose source targets or keys as predictors."""
    excluded = ["client_id", "cutoff", "y_any", *LABELS, "supervised_eligible"]
    predictors = [c for c in columns if c not in excluded]
    return {
        "keys": ["client_id", "cutoff"], "target": "y_any", "excluded_from_predictors": excluded,
        "predictors": predictors,
        "groups": {s: [c for c in predictors if c.startswith(s + '_')] for s in SOURCES},
        "ablation_groups": {
            "trx": [c for c in predictors if c.startswith('trx_')],
            "source_metadata": [c for c in predictors if c.startswith(('geo_', 'dialog_'))
                                and 'geohash_' not in c and 'embedding_' not in c],
            "geo_content": [c for c in predictors if c.startswith('geo_') and 'geohash_' in c],
            "dialog_content": [c for c in predictors if c.startswith('dialog_') and 'embedding_' in c],
            "cohort_flags": ["cold_start"],
        },
        "missing_numeric": "Keep NaN; counts are zero only within calendar overlap with the source",
        "currency_sidecar": "Separate units; do not sum amounts across currency values",
        "scope_note": "Calendar extract overlap is not continuous observation of each customer",
    }


In [ ]:
from dataclasses import replace

CONFIG_PATH = PROJECT_ROOT / "configs" / "propensity_data.local.json"
if not CONFIG_PATH.is_file():
    CONFIG_PATH = PROJECT_ROOT / "configs" / "propensity_data.json"

config = load_config(CONFIG_PATH)
# Có thể sửa trực tiếp tại đây trên máy thực thi, ví dụ:
# config = replace(config, data_root=Path("D:/Datasets/MBD-mini"),
#                  output_dir=PROJECT_ROOT / "outputs" / "propensity_notebook_v1")
# config = replace(config, include_dialog_embeddings=True)
display(pd.Series(config.to_dict(), name="configuration"))


## 3. Hàm đọc/ghi và kiểm tra chất lượng
Đọc raw ở chế độ read-only; mọi sản phẩm ghi vào output mới. SQL path được escape. Các helper kiểm tra khóa sẽ dừng khi phát hiện lỗi, không tự sửa nhãn hay xóa khách.


In [ ]:
"""Read-only source discovery, SQL quoting and output helpers."""
import json
from pathlib import Path


def literal(value) -> str:
    return "'" + str(value).replace("'", "''") + "'"


def write_json(path: Path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def files_at(root: Path) -> list[Path]:
    files = sorted(root.rglob("*.parquet")) if root.is_dir() else [root] if root.is_file() else []
    if not files:
        raise FileNotFoundError(f"No Parquet files under {root}")
    return files


def parquet_scan(files) -> str:
    paths = ",".join(literal(p.as_posix()) for p in files)
    return (f"read_parquet([{paths}], hive_partitioning=false, union_by_name=false, "
            "filename=true, file_row_number=true)")


def export(con, query: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"COPY ({query}) TO {literal(path.as_posix())} (FORMAT PARQUET, COMPRESSION ZSTD)")


def assert_empty(con, query: str, message: str):
    if con.execute(f"SELECT EXISTS ({query})").fetchone()[0]:
        raise ValueError(message)


def normalized_id(column="client_id"):
    return f"nullif(trim(cast({column} AS VARCHAR)), '')"


### Lịch đánh giá và thời điểm nhãn sẵn có
Validation cuốn chiếu 09–11/2022; fit cuối 02–11, calibration 12/2022, test khóa 01/2023. Đây là giả định nhãn sẵn ngay khi tháng kết thúc. `label_delay_days` là kịch bản giả định; lịch không khả thi sẽ bị từ chối. Không refit base model sau calibration.


In [ ]:
"""Offline evaluation calendar with explicit assumed label availability."""
from datetime import date, timedelta



def next_month(day: date) -> date:
    return date(day.year + (day.month == 12), day.month % 12 + 1, 1)


def evaluation_protocol(config, months: list[str]) -> dict:
    expected = []
    day = month_start(config.first_month)
    while day <= month_start(config.last_month):
        expected.append(day.strftime("%Y-%m"))
        day = next_month(day)
    if months != expected:
        raise ValueError("Targets do not cover every configured month")
    available = {
        m: next_month(month_start(m)) + timedelta(days=config.label_delay_days)
        for m in months
    }

    def training_at(cutoff_month):
        cutoff = month_start(cutoff_month)
        result = [m for m in months if m < cutoff_month and available[m] <= cutoff]
        if not result:
            raise ValueError(f"No labels available to train at {cutoff_month}")
        return result

    calibration = config.calibration_month
    test = config.test_month
    # Do not silently retain an impossible calibration schedule when delay > 0.
    if available[calibration] > month_start(test):
        raise ValueError("Calibration labels unavailable at test cutoff; choose an earlier calibration month")
    for m in config.validation_months:
        if available[m] > month_start(calibration):
            raise ValueError("Validation labels unavailable at final base-model fit")
    return {
        "timezone": "UTC",
        "feature_rule": "event_time < cutoff; fixed-day windows are [cutoff - days, cutoff)",
        "label_availability": "Assumption, not verified publisher SLA",
        "label_delay_days": config.label_delay_days,
        "assumed_label_available_at": {m: d.isoformat() + "T00:00:00Z" for m, d in available.items()},
        "rolling_validation": [{"train": training_at(m), "validate": m} for m in config.validation_months],
        "final_train": training_at(calibration),
        "base_model_fit_at": calibration + "-01T00:00:00Z",
        "calibration": calibration,
        "test": test,
        "refit_after_calibration": False,
        "primary_budget_fraction": 0.01,
        "secondary_budget_fractions": [0.005, 0.05],
        "metrics": ["recall_at_k", "precision_at_k", "lift_at_k", "average_precision",
                    "roc_auc", "log_loss", "brier", "calibration"],
        "bootstrap_unit": "client_id",
        "note": "Existing-customer temporal evaluation; never use client_id or source fold as predictors.",
    }


### Chuẩn hóa nhãn
- Có ít nhất một nhãn 1 → `y_any=1`, kể cả nhãn khác null.
- Đủ bốn nhãn 0 → `y_any=0`.
- Không có 1 và còn nhãn null → chưa xác định `y_any`.
- Mất cả bốn target hoặc thiếu dòng target → không dùng supervised; không điền 0.

Kiểm tra trùng khách–tháng sau chuẩn hóa `mon`. Mỗi tháng cấu hình cần ít nhất một dòng target để xác nhận nguồn có tháng đó. Quần thể scoring offline là toàn bộ ID trong `client_split`; không suy ra ngày mở tài khoản từ ngày đầu thấy log.


In [ ]:
def prepare_references(con, scans, config, out):
    con.execute(f"CREATE TABLE clients AS SELECT {normalized_id()} client_id FROM {scans['clients']}")
    assert_empty(con, "SELECT client_id FROM clients GROUP BY client_id HAVING client_id IS NULL OR count(*) > 1",
                 "client_split has missing or duplicate IDs")
    label_select = ", ".join(f"try_cast({c} AS DOUBLE) AS {c}" for c in LABELS)
    # Invalid source strings must not be turned silently into missing labels.
    invalid = " OR ".join(f"({c} IS NOT NULL AND (try_cast({c} AS DOUBLE) IS NULL "
                             f"OR try_cast({c} AS DOUBLE) NOT IN (0,1)))" for c in LABELS)
    assert_empty(con, f"SELECT 1 FROM {scans['targets']} WHERE {invalid}", "Targets contain values outside 0/1/null")
    con.execute(f"""CREATE TABLE targets AS
        WITH parsed AS (
            SELECT {normalized_id()} client_id, date_trunc('month', try_cast(mon AS TIMESTAMP)) cutoff,
                   cast(mon AS VARCHAR) raw_mon, {label_select},
                   filename source_file, file_row_number source_row
            FROM {scans['targets']}
        ) SELECT *, CASE
            WHEN {' OR '.join(c + ' = 1' for c in LABELS)} THEN 1
            WHEN {' AND '.join(c + ' = 0' for c in LABELS)} THEN 0
            ELSE NULL END::TINYINT y_any FROM parsed""")
    assert_empty(con, "SELECT 1 FROM targets WHERE client_id IS NULL OR cutoff IS NULL OR NOT isfinite(cutoff)",
                 "Invalid target ID or reporting month")
    assert_empty(con, "SELECT client_id, cutoff FROM targets GROUP BY ALL HAVING count(*) > 1", "Duplicate customer-month target keys")
    assert_empty(con, "SELECT 1 FROM targets t ANTI JOIN clients c USING(client_id)", "Target ID absent from client_split")
    # Targets can be incomplete: do not fabricate negative labels for absent rows.
    # All published client IDs form the offline scoring universe for every month.
    con.execute(f"CREATE VIEW selected_targets AS SELECT * FROM targets WHERE "
                f"cutoff >= TIMESTAMP {literal(config.first_month + '-01')} AND "
                f"cutoff <= TIMESTAMP {literal(config.last_month + '-01')}")
    months = [r[0] for r in con.execute("SELECT DISTINCT strftime(cutoff, '%Y-%m') FROM selected_targets ORDER BY 1").fetchall()]
    protocol = evaluation_protocol(config, months)
    export(con, "SELECT * FROM targets WHERE y_any IS NULL", out / "audit/unknown_labels.parquet")
    export(con, "SELECT cutoff, count(*) target_rows, count(y_any) labelled_rows, sum(y_any) positives "
           "FROM selected_targets GROUP BY cutoff ORDER BY cutoff", out / "audit/labels_by_month.parquet")
    return months, protocol


### Chuẩn hóa sự kiện
ID/time lỗi hoặc ID ngoài quần thể được đưa vào quarantine kèm file/số dòng. Thuộc tính phụ thiếu vẫn giữ event metadata. Không tự deduplicate từ timestamp trùng.

Amount không hữu hạn/không parse được thành null kèm cờ invalid; amount âm hữu hạn được giữ. Currency 11 là mã chính theo quy ước dự án; currency khác có giá trị mang cờ ngoại tệ. Currency thiếu/invalid tách riêng. Không gộp amount khác currency.

Embedding chỉ hợp lệ khi đúng số chiều và toàn bộ giá trị hữu hạn; embedding lỗi không làm mất log hội thoại.


In [ ]:
def normalize_events(con, name, scan, config, out):
    """Keep good events even when an optional attribute is absent/invalid."""
    fields = [f"{normalized_id()} AS client_id", "try_cast(event_time AS TIMESTAMP) event_time",
              "cast(event_time AS VARCHAR) raw_event_time", "filename source_file", "file_row_number source_row"]
    if name == "trx":
        fields += ["try_cast(amount AS DOUBLE) raw_amount", "cast(amount AS VARCHAR) amount_source",
                   "cast(currency AS VARCHAR) currency_source", "try_cast(currency AS DOUBLE) currency_number"]
        fields += [f"coalesce(cast({c} AS VARCHAR), '__MISSING__') AS {c}" for c in CATEGORIES]
    elif name == "geo":
        fields += [f"cast(geohash_{i} AS VARCHAR) geohash_{i}" for i in (4, 5, 6)]
    elif config.include_dialog_embeddings:
        fields += ["try_cast(embedding AS DOUBLE[]) raw_embedding"]
    con.execute(f"CREATE VIEW raw_{name} AS SELECT {', '.join(fields)} FROM {scan}")
    con.execute(f"""CREATE VIEW classified_{name} AS SELECT *, CASE
        WHEN client_id IS NULL THEN 'missing_client_id'
        WHEN event_time IS NULL OR NOT isfinite(event_time) THEN 'invalid_event_time'
        WHEN NOT EXISTS (SELECT 1 FROM clients c WHERE c.client_id = r.client_id) THEN 'unknown_client_id'
        ELSE NULL END AS rejection_reason FROM raw_{name} r""")
    export(con, f"SELECT * FROM classified_{name} WHERE rejection_reason IS NOT NULL",
           out / f"quarantine/{name}.parquet")
    additions = ""
    if name == "trx":
        known_currency = ("currency_number IS NOT NULL AND isfinite(currency_number) AND "
                          "currency_number = trunc(currency_number) AND try_cast(currency_number AS BIGINT) IS NOT NULL")
        additions = f""",
            CASE WHEN isfinite(raw_amount) THEN raw_amount END AS amount,
            amount_source IS NULL AS amount_missing,
            amount_source IS NOT NULL AND (raw_amount IS NULL OR NOT isfinite(raw_amount)) AS amount_invalid,
            CASE WHEN {known_currency} THEN try_cast(currency_number AS BIGINT) END AS currency,
            currency_source IS NULL AS currency_missing,
            currency_source IS NOT NULL AND NOT coalesce(({known_currency}), false) AS currency_invalid"""
    elif name == "dialog" and config.include_dialog_embeddings:
        valid = (f"array_length(raw_embedding) = {config.embedding_size} AND "
                 "array_length(list_filter(raw_embedding, x -> x IS NULL OR NOT isfinite(x))) = 0")
        additions = f", CASE WHEN {valid} THEN raw_embedding END AS embedding"
    con.execute(f"CREATE TABLE {name} AS SELECT * EXCLUDE(rejection_reason){additions} "
                f"FROM classified_{name} WHERE rejection_reason IS NULL")
    con.execute(f"ANALYZE {name}")
    counts = con.execute(f"SELECT count(*), count(*) FILTER(WHERE rejection_reason IS NOT NULL) FROM classified_{name}").fetchone()
    # A timestamp collision is reported, never treated as proof of duplication.
    export(con, f"SELECT client_id, event_time, count(*) n_events FROM {name} "
           "GROUP BY client_id, event_time HAVING count(*) > 1", out / f"audit/{name}_timestamp_collisions.parquet")
    start = con.execute(f"SELECT min(event_time) FROM {name}").fetchone()[0]
    return {"raw_rows": counts[0], "quarantined_rows": counts[1], "accepted_rows": counts[0] - counts[1],
            "first_valid_event_utc": start.isoformat() if start else None,
            "automatic_deduplication": False}


## 4. Định nghĩa đặc trưng trước cutoff
Mỗi nguồn có cửa sổ 30 ngày, 84 ngày và toàn lịch sử, với điều kiện `event_time < cutoff`. Cửa sổ ngày bao gồm biên trái, loại biên phải.

Trong phạm vi nguồn mà không có log: count 0; recency/mean/embedding null. Cửa sổ trước phạm vi nguồn: count null, overlap 0 và cờ truncated. Phạm vi theo lịch không chứng minh quan sát liên tục từng khách. Không dùng timestamp cuối tương lai để quyết định cohort.

Giữ cực trị và heavy users. Dùng log1p count/absolute sum và signed-log amount. Currency và tần suất event type/subtype xuất riêng ở sidecar; vocabulary/cap/PCA sau này chỉ fit trên train.


In [ ]:
def window_features(con, source, cutoff, days, config, out):
    """Aggregate one source/window before joining to the customer universe."""
    tag = "all" if days is None else f"{days}d"
    prefix = f"{source}_{tag}_"
    end = f"TIMESTAMP {literal(cutoff)}"
    lower = "" if days is None else f"AND event_time >= {end} - INTERVAL {days} DAY"
    con.execute(f"CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM {source} WHERE event_time < {end} {lower}")
    scope_start = con.execute("SELECT first_valid_event FROM source_bounds WHERE source = ?", [source]).fetchone()[0]
    start = f"TIMESTAMP {literal(scope_start)}" if scope_start else "NULL::TIMESTAMP"
    # Use source start only: a future last-event timestamp must not define coverage.
    overlap_start = start if days is None else f"greatest({start}, {end} - INTERVAL {days} DAY)"
    overlap = f"CASE WHEN {start} IS NULL THEN 0 ELSE greatest(0, epoch({end} - {overlap_start}) / 86400.0) END"
    truncated = (f"({start} IS NULL OR {start} >= {end})" if days is None else
                 f"({start} IS NULL OR {start} > {end} - INTERVAL {days} DAY)")
    numeric = {
        "event_count": "count(*)", "active_days": "count(DISTINCT cast(event_time AS DATE))",
        "days_since_last_event": f"epoch({end} - max(event_time)) / 86400.0",
        "days_since_first_recorded_event": f"epoch({end} - min(event_time)) / 86400.0",
    }
    zero_when_covered = {"event_count", "active_days"}
    if source == "trx":
        numeric.update({
            "main_count": "count(*) FILTER(WHERE currency = 11)",
            "foreign_count": "count(*) FILTER(WHERE currency <> 11)",
            "currency_missing_count": "count(*) FILTER(WHERE currency_missing)",
            "currency_invalid_count": "count(*) FILTER(WHERE currency_invalid)",
            "amount_missing_count": "count(*) FILTER(WHERE amount_missing)",
            "amount_invalid_count": "count(*) FILTER(WHERE amount_invalid)",
            "main_amount_count": "count(amount) FILTER(WHERE currency = 11)",
            "main_negative_amount_count": "count(*) FILTER(WHERE currency = 11 AND amount < 0)",
            "main_amount_sum": "sum(amount) FILTER(WHERE currency = 11)",
            "main_amount_mean": "avg(amount) FILTER(WHERE currency = 11)",
            "main_amount_median": "median(amount) FILTER(WHERE currency = 11)",
            "main_amount_max": "max(amount) FILTER(WHERE currency = 11)",
            "main_amount_abs_sum": "sum(abs(amount)) FILTER(WHERE currency = 11)",
            "main_amount_signed_log_mean": "avg(sign(amount) * ln(1 + abs(amount))) FILTER(WHERE currency = 11)",
            "foreign_fraction": "count(*) FILTER(WHERE currency <> 11)::DOUBLE / count(*)",
        })
        zero_when_covered.update(k for k in numeric if k.endswith("_count"))
        for cat in CATEGORIES:
            numeric[cat + "_nunique"] = f"count(DISTINCT {cat}) FILTER(WHERE {cat} <> '__MISSING__')"
            numeric[cat + "_missing_count"] = f"count(*) FILTER(WHERE {cat} = '__MISSING__')"
            zero_when_covered.update((cat + "_nunique", cat + "_missing_count"))
        # Sidecar keeps each valid currency as its own monetary unit.
        export(con, "SELECT client_id, currency, count(*) event_count, count(amount) amount_count, "
               "sum(amount) amount_sum, median(amount) amount_median, max(amount) amount_max "
               "FROM events WHERE currency IS NOT NULL GROUP BY client_id, currency",
               out / f"currency/month={cutoff[:7]}/{tag}.parquet")
        # Long-form frequencies avoid learning a category vocabulary from future months.
        category_queries = [
            f"SELECT client_id, {literal(cat)} attribute, {cat} category, count(*) event_count "
            f"FROM events GROUP BY client_id, {cat}" for cat in ("event_type", "event_subtype")
        ]
        export(con, " UNION ALL ".join(category_queries),
               out / f"categories/month={cutoff[:7]}/{tag}.parquet")
    elif source == "geo":
        for i in (4, 5, 6):
            numeric[f"geohash_{i}_nunique"] = f"count(DISTINCT geohash_{i})"
            numeric[f"geohash_{i}_missing_count"] = f"count(*) FILTER(WHERE geohash_{i} IS NULL)"
            zero_when_covered.update((f"geohash_{i}_nunique", f"geohash_{i}_missing_count"))
    expressions = ", ".join(f"{expression} AS {name}" for name, expression in numeric.items())
    con.execute(f"CREATE OR REPLACE TEMP TABLE aggregate_window AS SELECT client_id, {expressions} FROM events GROUP BY client_id")
    select = ["c.client_id", f"({overlap}) AS {prefix}source_window_overlap_days",
              f"cast({truncated} AS TINYINT) AS {prefix}history_truncated",
              f"cast(a.event_count IS NOT NULL AS TINYINT) AS {prefix}has_event"]
    for key in numeric:
        expression = (f"CASE WHEN ({overlap}) > 0 THEN coalesce(a.{key}, 0) END"
                      if key in zero_when_covered else f"a.{key}")
        select.append(f"{expression} AS {prefix}{key}")
    select.append(f"CASE WHEN ({overlap}) > 0 THEN ln(1 + coalesce(a.event_count, 0)) END "
                  f"AS {prefix}log1p_event_count")
    if source == "trx":
        select += [f"ln(1 + a.main_amount_abs_sum) AS {prefix}log1p_main_amount_abs_sum",
                   f"CASE WHEN ({overlap}) > 0 THEN cast(coalesce(a.foreign_count, 0) > 0 AS TINYINT) END "
                   f"AS {prefix}has_foreign"]
    query = f"SELECT {', '.join(select)} FROM clients c LEFT JOIN aggregate_window a USING(client_id)"
    con.execute(f"CREATE OR REPLACE TEMP TABLE window_result AS {query}")
    if source == "trx":
        for cat in ("event_type", "event_subtype"):
            con.execute(f"CREATE OR REPLACE TEMP TABLE category_frequency AS SELECT client_id, {cat}, count(*) n "
                        f"FROM events WHERE {cat} <> '__MISSING__' GROUP BY client_id, {cat}")
            con.execute(f"ALTER TABLE window_result ADD COLUMN {prefix}{cat}_top_share DOUBLE")
            con.execute(f"UPDATE window_result w SET {prefix}{cat}_top_share = g.share FROM "
                        "(SELECT client_id, max(n)::DOUBLE/sum(n) share FROM category_frequency GROUP BY client_id) g "
                        "WHERE w.client_id = g.client_id")
    if source == "geo":
        con.execute("CREATE OR REPLACE TEMP TABLE geo_frequency AS SELECT client_id, geohash_6, count(*) n "
                    "FROM events WHERE geohash_6 IS NOT NULL GROUP BY client_id, geohash_6")
        con.execute(f"ALTER TABLE window_result ADD COLUMN {prefix}top_geohash_6_share DOUBLE")
        con.execute(f"UPDATE window_result w SET {prefix}top_geohash_6_share = g.share FROM "
                    "(SELECT client_id, max(n)::DOUBLE/sum(n) share FROM geo_frequency GROUP BY client_id) g "
                    "WHERE w.client_id = g.client_id")
    if source == "dialog" and config.include_dialog_embeddings and days == config.embedding_window_days:
        expr = ["count(embedding) AS valid_embedding_count"]
        expr += [f"avg(embedding[{i}]) AS mean_{i}" for i in range(1, config.embedding_size + 1)]
        expr += ["first(embedding ORDER BY event_time DESC, source_file, source_row) "
                 "FILTER(WHERE embedding IS NOT NULL) AS latest"]
        con.execute(f"CREATE OR REPLACE TEMP TABLE dialog_vectors AS SELECT client_id, {', '.join(expr)} "
                    "FROM events GROUP BY client_id")
        cols = ["w.*", f"CASE WHEN w.{prefix}source_window_overlap_days > 0 "
                f"THEN coalesce(v.valid_embedding_count, 0) END AS {prefix}valid_embedding_count"]
        cols += [f"v.mean_{i} AS {prefix}embedding_mean_{i}" for i in range(1, config.embedding_size + 1)]
        cols += [f"v.latest[{i}] AS {prefix}embedding_last_{i}" for i in range(1, config.embedding_size + 1)]
        con.execute(f"CREATE OR REPLACE TEMP TABLE with_vectors AS SELECT {', '.join(cols)} "
                    "FROM window_result w LEFT JOIN dialog_vectors v USING(client_id)")
        con.execute("DROP TABLE window_result")
        con.execute("ALTER TABLE with_vectors RENAME TO window_result")
    return prefix


### Ghép về khách–tháng
Tổng hợp riêng từng nguồn rồi left join vào khung khách. Kiểm tra không mất khách hoặc nhân số dòng. `cold_start=1` khi không nguồn nào có event trước cutoff. Whitelist predictor loại ID, cutoff, nhãn nguồn, `y_any` và cờ eligibility.


In [ ]:
def build_month(con, month, config, out):
    cutoff = month + "-01 00:00:00"
    con.execute(f"""CREATE OR REPLACE TEMP TABLE features AS
        SELECT c.client_id, TIMESTAMP {literal(cutoff)} cutoff,
               t.y_any, {', '.join('t.' + c for c in LABELS)},
               cast(t.y_any IS NOT NULL AS BOOLEAN) supervised_eligible
        FROM clients c LEFT JOIN selected_targets t
          ON c.client_id = t.client_id AND t.cutoff = TIMESTAMP {literal(cutoff)}""")
    for source in SOURCES:
        for days in (*config.windows_days, None):
            window_features(con, source, cutoff, days, config, out)
            con.execute("CREATE OR REPLACE TEMP TABLE joined AS SELECT f.*, w.* EXCLUDE(client_id) "
                        "FROM features f LEFT JOIN window_result w USING(client_id)")
            con.execute("DROP TABLE features")
            con.execute("ALTER TABLE joined RENAME TO features")
        con.execute(f"ALTER TABLE features ADD COLUMN {source}_ever_seen_source TINYINT")
        con.execute(f"UPDATE features SET {source}_ever_seen_source = {source}_all_has_event")
        if source == "dialog":
            con.execute("ALTER TABLE features ADD COLUMN dialog_has_dialog TINYINT")
            con.execute("UPDATE features SET dialog_has_dialog = dialog_ever_seen_source")
    con.execute("ALTER TABLE features ADD COLUMN cold_start TINYINT")
    con.execute("UPDATE features SET cold_start = cast(trx_ever_seen_source = 0 AND "
                "geo_ever_seen_source = 0 AND dialog_ever_seen_source = 0 AS TINYINT)")
    assert_empty(con, "SELECT client_id FROM features GROUP BY client_id HAVING count(*) > 1", "Join multiplied customer rows")
    expected = con.execute("SELECT count(*) FROM clients").fetchone()[0]
    actual, known, positive, cold = con.execute("SELECT count(*), count(y_any), coalesce(sum(y_any),0), "
                                               "sum(cold_start) FROM features").fetchone()
    if actual != expected:
        raise ValueError("Feature preparation lost customers")
    export(con, "SELECT * FROM features ORDER BY client_id", out / f"features/month={month}/part.parquet")
    columns = [row[0] for row in con.execute("DESCRIBE features").fetchall()]
    return {"month": month, "rows": actual, "labelled_rows": known, "positives": positive,
            "unknown_labels": actual - known, "cold_start_rows": cold}, columns


## 5. Bắt đầu một lần chuẩn bị dữ liệu
**Từ cell này trở đi sẽ đọc raw và ghi output khi người dùng thực thi.** Các cell trước chỉ định nghĩa mã và đọc cấu hình.

Inventory dùng path/size/mtime để phát hiện thay đổi trong lần chạy; không phải checksum byte raw. Output đã tồn tại sẽ bị từ chối, không tự ghi đè hoặc resume.


In [ ]:
sources = {"clients": "client_split", "targets": "targets", **SOURCES}
files = {key: files_at(config.data_root / rel) for key, rel in sources.items()}
inventory = [
    {"source": key, "path": p.relative_to(config.data_root).as_posix(),
     "bytes": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns}
    for key, paths in files.items() for p in paths
]
fingerprint = hashlib.sha256(json.dumps(inventory, sort_keys=True).encode()).hexdigest()

out = config.output_dir
out.mkdir(parents=True, exist_ok=False)
manifest = {
    "status": "running",
    "started_at": datetime.now(timezone.utc).isoformat(),
    "entrypoint": "notebooks/02_data_preparation.ipynb",
    "config": config.to_dict(),
    "source_inventory_sha256": fingerprint,
    "fingerprint_note": "Paths, sizes and mtimes; not a full byte checksum",
    "duckdb_version": duckdb.__version__,
    "models_fitted": False, "raw_modified": False,
    "amount_capping": False, "negative_sampling": False,
    "source_audits": {}, "months": [],
}
con = None
write_json(out / "manifest.json", manifest)
write_json(out / "source_inventory.json", inventory)

@contextmanager
def notebook_stage(name):
    if manifest["status"] != "running":
        raise RuntimeError("Lần chạy đã kết thúc hoặc lỗi; dùng output mới và Restart Kernel.")
    try:
        yield
    except BaseException as exc:
        manifest["status"] = "failed"
        manifest["failed_stage"] = name
        manifest["error"] = f"{type(exc).__name__}: {exc}"
        try:
            if con is not None:
                con.close()
        finally:
            write_json(out / "manifest.json", manifest)
        raise


In [ ]:
with notebook_stage("open_database"):
    con = duckdb.connect(str(out / "work.duckdb"))
    con.execute("SET TimeZone = 'UTC'")
    con.execute(f"SET memory_limit = {literal(config.memory_limit)}")
    con.execute(f"SET threads = {config.threads}")
    con.execute("SET preserve_insertion_order = false")
    con.execute(f"SET temp_directory = {literal((out / 'spill').as_posix())}")
    scans = {key: parquet_scan(paths) for key, paths in files.items()}


## 6. Kiểm tra nhãn và ghi protocol
Bảng kiểm kê có riêng số dòng mất cả bốn target. Đây là kiểm tra trước khi dựng feature; quyết định dùng supervised nằm ở `supervised_eligible` sau khi nối về đầy đủ khách–tháng.


In [ ]:
with notebook_stage("prepare_references"):
    months, protocol = prepare_references(con, scans, config, out)
    write_json(out / "protocol.json", protocol)
    con.execute("CREATE TABLE source_bounds (source VARCHAR PRIMARY KEY, first_valid_event TIMESTAMP)")
    label_audit = con.execute("""
        SELECT cutoff, count(*) target_rows, count(y_any) labelled_rows,
               count(*) FILTER (WHERE target_1 IS NULL AND target_2 IS NULL
                                  AND target_3 IS NULL AND target_4 IS NULL) all_four_missing,
               count(*) FILTER (WHERE y_any IS NULL) unknown_y_any,
               sum(y_any) positives
        FROM selected_targets GROUP BY cutoff ORDER BY cutoff
    """).df()
    display(label_audit)
    display(pd.Series(protocol, name="offline_protocol"))


## 7. Chuẩn hóa lần lượt từng nguồn
Chạy mỗi cell một lần trong cùng session. DuckDB lưu bản chuẩn hóa và có thể spill ra đĩa. Audit timestamp collision chỉ báo ứng viên; không loại giao dịch. Cột thuộc tính phụ chưa xác minh trong targets không được dùng làm feature.


In [ ]:
def normalize_one_source(source):
    with notebook_stage("normalize_" + source):
        audit = normalize_events(con, source, scans[source], config, out)
        manifest["source_audits"][source] = audit
        con.execute("INSERT INTO source_bounds VALUES (?, ?)",
                    [source, audit["first_valid_event_utc"]])
        write_json(out / "manifest.json", manifest)
        return pd.Series(audit, name=source)


In [ ]:
display(normalize_one_source("trx"))


In [ ]:
display(normalize_one_source("geo"))


In [ ]:
display(normalize_one_source("dialog"))


## 8. Tạo bảng đặc trưng theo tháng
Giữ toàn bộ khách, gồm cold-start và dòng chưa xác định nhãn. Không chọn cohort theo hoạt động tương lai. Schema phải thống nhất giữa các tháng; từng tháng được ghi Parquet ngay để không giữ cả ma trận trong RAM.


In [ ]:
feature_columns = None
for month in months:
    with notebook_stage("features_" + month):
        summary, columns = build_month(con, month, config, out)
        if feature_columns is not None and columns != feature_columns:
            raise ValueError("Feature schema changed across months")
        feature_columns = columns
        manifest["months"].append(summary)
        write_json(out / "manifest.json", manifest)
        print(summary)

schema = feature_schema(feature_columns)
display(pd.DataFrame(manifest["months"]))


## 9. Tách dòng có nhãn khi đưa vào mô hình
File features giữ toàn bộ quần thể scoring. Hàm dưới chỉ trả các batch có `supervised_eligible=True` và chỉ lấy predictor theo whitelist, khóa và `y_any`. Do đó dòng mất cả bốn target không đi vào train/evaluation; các tháng có nhãn khác của cùng khách vẫn dùng được.

Hàm là ví dụ đọc theo batch cho bước mô hình sau này; định nghĩa hàm không nạp ma trận hoặc fit mô hình. Không dùng random split, không dùng tỷ lệ mua từ tháng đang dự đoán để fit fallback.


In [ ]:
def iter_supervised_batches(month, batch_size=10_000):
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError("batch_size must be a positive integer")
    if month not in months:
        raise ValueError("Month outside this prepared run")
    columns = [*schema["keys"], *schema["predictors"], schema["target"]]
    path = out / f"features/month={month}/part.parquet"
    reader = pq.ParquetFile(path)
    for batch in reader.iter_batches(
        batch_size=batch_size, columns=[*columns, "supervised_eligible"]
    ):
        eligible = pc.equal(batch.column("supervised_eligible"), True)
        selected = batch.filter(eligible).select(columns)
        if selected.num_rows:
            yield selected

# Sau khi manifest ready, bước mô hình có thể dùng:
# for batch in iter_supervised_batches(protocol["final_train"][0]):
#     ...  # downstream training/incremental loading; không thực thi trong notebook này


## 10. Đối soát và hoàn tất
Chỉ dùng output khi manifest có `status=ready`. Kiểm tra raw inventory không đổi, ghi whitelist và đóng database. Không fit scaler, PCA, classifier hoặc calibrator trong notebook này.


In [ ]:
with notebook_stage("finalize"):
    final_inventory = [
        {"source": key, "path": p.relative_to(config.data_root).as_posix(),
         "bytes": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns}
        for key, rel in sources.items() for p in files_at(config.data_root / rel)
    ]
    if final_inventory != inventory:
        raise ValueError("Raw source inventory changed during preparation")
    write_json(out / "feature_schema.json", schema)
    con.close()
    manifest["status"] = "ready"
    manifest["finished_at"] = datetime.now(timezone.utc).isoformat()
    write_json(out / "manifest.json", manifest)
print(out / "manifest.json")


## Đầu ra và giới hạn
- `features/month=YYYY-MM/part.parquet`: đầy đủ khách–tháng và nhãn/eligibility.
- `currency/`, `categories/`: tổng hợp riêng currency và tần suất mã giao dịch.
- `quarantine/`, `audit/`: dòng lỗi ID/time, nhãn và timestamp collisions.
- `protocol.json`, `feature_schema.json`, `source_inventory.json`, `manifest.json`.
- `work.duckdb`, `spill/`: bản chuẩn hóa và vùng tạm; không đưa lên Git.

Không kết luận mô hình tốt trước khi chạy validation. So sánh trx → thêm metadata nguồn → geo → dialog trên cùng cohort, lịch và ngân sách. Báo cáo Recall/Precision/Lift@1%, AP và calibration, có nhóm cold-start.

Xem [hướng dẫn chi tiết](../docs/propensity_data_preparation.md) và [thiết kế any-product](../docs/propensity_feasibility/any_product/design.md). Mã trong các cell có `source_file` được đồng bộ từ helper tương ứng; công cụ `scripts/sync_preparation_notebook.py` chỉ cập nhật nội dung cell, không thực thi notebook.
